In [ ]:
# Real: 1720
# Fake: 1720
# Total: 3440

In [ ]:
# Real:
# challenge           500
# whichfaceisreal     600
# hidf                300
# augreal             200
# hardreal            120

# Fake:
# challenge           500
# faceswap            340
# deepfacelab         270
# simswap             225
# e4s                 120
# inswap               65
# degradedfake        200

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Cell 2 — Imports, paths, and target counts

# import os
# import cv2
# import glob
# import json
# import random
# import shutil
# import zipfile
# import hashlib
# import numpy as np
# from pathlib import Path
# from tqdm.auto import tqdm
# from collections import defaultdict

# random.seed(42)
# np.random.seed(42)

# # =========================
# # Main paths
# # =========================
# BASE_DRIVE = "/content/drive/MyDrive/Capstone_Hripsime"

# TRAIN_MIX_DIR = Path(f"{BASE_DRIVE}/Database/forensichub_train_mix")
# TMP_DIR = Path("/content/tmp_forensichub_extract")
# STAGING = Path("/content/DiffusionForensics_Recons")

# TRAIN_MIX_DIR.mkdir(parents=True, exist_ok=True)
# TMP_DIR.mkdir(parents=True, exist_ok=True)
# STAGING.mkdir(parents=True, exist_ok=True)

# # Original challenge dataset
# CHALLENGE_TRAIN_DIR = Path(f"{BASE_DRIVE}/Database/RDDC_Dataset/Datasets/training_data")
# VAL_DIR = Path(f"{BASE_DRIVE}/Database/RDDC_Dataset/Datasets/validation_data")
# TEST_DIR = Path(f"{BASE_DRIVE}/Database/RDDC_Dataset/Datasets/publictest_data")

# VAL_REF = Path(f"{BASE_DRIVE}/Database/RDDC_Dataset/RDDC_References/validation_reference/reference.txt")
# TEST_REF = Path(f"{BASE_DRIVE}/Database/CRDDC_Dataset/RDDC_References/publictest_reference/reference.txt")

# # External real sources
# WHICHFACEISREAL_ZIP = Path(f"{BASE_DRIVE}/Database/Data Expansion Resources/Copy of whichfaceisreal.zip")
# HIDF_REAL_DIR = Path(f"{BASE_DRIVE}/Database/Data Expansion Resources/hidf_files/real_images/Real-img")

# # External fake sources, mainly DF40-style manipulation methods
# FAKE_ZIPS = {
#     "faceswap": Path(f"{BASE_DRIVE}/Database/Data Expansion Resources/Copy of faceswap.zip"),
#     "deepfacelab": Path(f"{BASE_DRIVE}/Database/Data Expansion Resources/Copy of deepfacelab.zip"),
#     "simswap": Path(f"{BASE_DRIVE}/Database/Data Expansion Resources/Copy of simswap.zip"),
#     "e4s": Path(f"{BASE_DRIVE}/Database/Data Expansion Resources/Copy of e4s.zip"),
#     "inswap": Path(f"{BASE_DRIVE}/Database/Data Expansion Resources/Copy of inswap.zip"),
# }

# # Final target distribution
# TARGET_REAL = {
#     "challenge": 500,
#     "whichfaceisreal": 600,
#     "hidf": 300,
#     "augreal": 200,
#     "hardreal": 120,
# }

# TARGET_FAKE = {
#     "challenge": 500,
#     "faceswap": 340,
#     "deepfacelab": 270,
#     "simswap": 225,
#     "e4s": 120,
#     "inswap": 65,
#     "degradedfake": 200,
# }

# VALID_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")

In [ ]:
# Cell 3 — Helper functions

# def is_image_file(path):
#     return Path(path).suffix.lower() in VALID_EXTS


# def next_index(out_dir):
#     nums = []
#     for p in out_dir.glob("*"):
#         if p.is_file():
#             prefix = p.stem.split("_")[0]
#             if prefix.isdigit():
#                 nums.append(int(prefix))
#     return max(nums) + 1 if nums else 0


# def clear_folder(folder):
#     folder = Path(folder)
#     for p in folder.glob("*"):
#         if p.is_file():
#             p.unlink()


# def extract_zip(zip_path, extract_name):
#     zip_path = Path(zip_path)
#     extract_path = TMP_DIR / extract_name

#     if extract_path.exists():
#         shutil.rmtree(extract_path)

#     extract_path.mkdir(parents=True, exist_ok=True)

#     print(f"Extracting: {zip_path}")
#     with zipfile.ZipFile(zip_path, "r") as zf:
#         zf.extractall(extract_path)

#     return extract_path


# def collect_images_from_dir(root_dir):
#     root_dir = Path(root_dir)
#     paths = []

#     for root, _, files in os.walk(root_dir):
#         for f in files:
#             p = Path(root) / f
#             if is_image_file(p):
#                 paths.append(str(p))

#     return sorted(paths)


# def read_image_cv2(path):
#     img = cv2.imread(str(path))
#     return img


# def save_image_cv2(img, out_path):
#     out_path = Path(out_path)
#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     return cv2.imwrite(str(out_path), img)


# def save_sampled_images(paths, count, source_name, label_name, out_dir, seed=42):
#     """
#     Saves sampled images into format:
#     0000_source_label.jpg

#     label_name must be "real" or "fake".
#     """
#     out_dir = Path(out_dir)
#     paths = list(paths)

#     if len(paths) < count:
#         raise ValueError(f"Not enough images for {source_name}_{label_name}: found {len(paths)}, need {count}")

#     rng = random.Random(seed)
#     sampled = rng.sample(paths, count)

#     idx = next_index(out_dir)
#     saved = 0

#     for p in tqdm(sampled, desc=f"Saving {source_name}_{label_name}"):
#         img = cv2.imread(str(p))
#         if img is None:
#             continue

#         dst = out_dir / f"{idx:04d}_{source_name}_{label_name}.jpg"
#         ok = cv2.imwrite(str(dst), img)

#         if ok:
#             idx += 1
#             saved += 1

#     print(f"Saved {saved}/{count}: {source_name}_{label_name}")
#     return saved


# def count_distribution(data_dir):
#     data_dir = Path(data_dir)

#     real_counts = defaultdict(int)
#     fake_counts = defaultdict(int)

#     for p in data_dir.iterdir():
#         if not p.is_file():
#             continue

#         name = p.name.lower()
#         parts = name.split("_")
#         source = parts[1] if len(parts) >= 3 else "unknown"

#         if "_real" in name:
#             real_counts[source] += 1
#         elif "_fake" in name:
#             fake_counts[source] += 1

#     total_real = sum(real_counts.values())
#     total_fake = sum(fake_counts.values())

#     print("Real image distribution:\n")
#     for k, v in sorted(real_counts.items()):
#         print(f"{k:20s}: {v}")
#     print("\nTotal real images:", total_real)

#     print("\n" + "=" * 40 + "\n")

#     print("Fake image distribution:\n")
#     for k, v in sorted(fake_counts.items()):
#         print(f"{k:20s}: {v}")
#     print("\nTotal fake images:", total_fake)

#     print("\n" + "=" * 40 + "\n")
#     print("Overall dataset balance:")
#     print(f"Real: {total_real}")
#     print(f"Fake: {total_fake}")
#     print(f"Total: {total_real + total_fake}")

#     if total_real == total_fake:
#         print("Status: perfectly balanced")
#     else:
#         print(f"Status: off by {abs(total_real - total_fake)}")

#     return real_counts, fake_counts

In [ ]:
# Cell 4 — Optional: clear previous training mix

# WARNING: this deletes all files in forensichub_train_mix
# clear_folder(TRAIN_MIX_DIR)
# print("Cleared:", TRAIN_MIX_DIR)

In [ ]:
# Cell 5 — Add original challenge training images: 500 real + 500 fake

# challenge_paths = [
#     str(p) for p in CHALLENGE_TRAIN_DIR.iterdir()
#     if p.is_file() and is_image_file(p)
# ]

# challenge_real = [p for p in challenge_paths if "_real" in Path(p).name.lower()]
# challenge_fake = [p for p in challenge_paths if "_fake" in Path(p).name.lower()]

# print("Challenge real found:", len(challenge_real))
# print("Challenge fake found:", len(challenge_fake))

# save_sampled_images(
#     challenge_real,
#     TARGET_REAL["challenge"],
#     "challenge",
#     "real",
#     TRAIN_MIX_DIR,
#     seed=42
# )

# save_sampled_images(
#     challenge_fake,
#     TARGET_FAKE["challenge"],
#     "challenge",
#     "fake",
#     TRAIN_MIX_DIR,
#     seed=42
# )

# count_distribution(TRAIN_MIX_DIR)

In [ ]:
# Cell 6 — Add fake images from DF40-style manipulation sources

# def collect_fake_images_from_extracted_dir(root_dir, source_name):
#     """
#     Collect fake images from extracted fake-method folders.
#     Skips paths that clearly contain real/authentic names.
#     """
#     root_dir = Path(root_dir)
#     source_name = source_name.lower()

#     fake_paths = []

#     for root, _, files in os.walk(root_dir):
#         root_lower = root.lower()

#         # Avoid real folders if present
#         if "real" in root_lower or "authentic" in root_lower or "whichfaceisreal" in root_lower:
#             continue

#         for f in files:
#             p = Path(root) / f
#             if not is_image_file(p):
#                 continue

#             name_lower = p.name.lower()

#             # If filename explicitly says real, skip it
#             if "real" in name_lower or "authentic" in name_lower:
#                 continue

#             fake_paths.append(str(p))

#     return sorted(fake_paths)


# for source_name in ["faceswap", "deepfacelab", "simswap", "e4s", "inswap"]:
#     zip_path = FAKE_ZIPS[source_name]
#     target_count = TARGET_FAKE[source_name]

#     extracted = extract_zip(zip_path, source_name)
#     fake_paths = collect_fake_images_from_extracted_dir(extracted, source_name)

#     print(f"\n{source_name} fake found:", len(fake_paths))

#     save_sampled_images(
#         fake_paths,
#         target_count,
#         source_name,
#         "fake",
#         TRAIN_MIX_DIR,
#         seed=42
#     )

# count_distribution(TRAIN_MIX_DIR)

In [ ]:
# Cell 7 — Add real images from WhichFaceIsReal: 600 real

# extracted_wfir = extract_zip(WHICHFACEISREAL_ZIP, "whichfaceisreal")
# whichfaceisreal_paths = collect_images_from_dir(extracted_wfir)

# print("WhichFaceIsReal real found:", len(whichfaceisreal_paths))

# save_sampled_images(
#     whichfaceisreal_paths,
#     TARGET_REAL["whichfaceisreal"],
#     "whichfaceisreal",
#     "real",
#     TRAIN_MIX_DIR,
#     seed=42
# )

# count_distribution(TRAIN_MIX_DIR)

In [ ]:
# Cell 8 — Add HiDF real images: 300 real

# hidf_real_paths = collect_images_from_dir(HIDF_REAL_DIR)

# print("HiDF real found:", len(hidf_real_paths))

# save_sampled_images(
#     hidf_real_paths,
#     TARGET_REAL["hidf"],
#     "hidf",
#     "real",
#     TRAIN_MIX_DIR,
#     seed=42
# )

# count_distribution(TRAIN_MIX_DIR)

In [ ]:
# Cell 9 — Create augmented real images: 200 real

# def degrade_real_image(img):
#     """
#     Moderate real-image degradation used to make the model more robust
#     to blur, noise, darkness, and compression.
#     """
#     out = img.copy()

#     # Blur
#     if random.random() < 0.8:
#         k = random.choice([3, 5, 7])
#         out = cv2.GaussianBlur(out, (k, k), 0)

#     # Darkening
#     if random.random() < 0.7:
#         alpha = random.uniform(0.55, 0.9)
#         out = np.clip(out.astype(np.float32) * alpha, 0, 255).astype(np.uint8)

#     # Noise
#     if random.random() < 0.7:
#         noise = np.random.normal(0, random.uniform(6, 18), out.shape).astype(np.float32)
#         out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)

#     # JPEG compression
#     if random.random() < 0.9:
#         q = random.randint(25, 55)
#         ok, enc = cv2.imencode(".jpg", out, [int(cv2.IMWRITE_JPEG_QUALITY), q])
#         if ok:
#             out = cv2.imdecode(enc, cv2.IMREAD_COLOR)

#     return out


# def create_augmented_reals(target_count=200):
#     source_paths = [
#         p for p in TRAIN_MIX_DIR.iterdir()
#         if p.is_file()
#         and "_real" in p.name.lower()
#         and "_augreal_real" not in p.name.lower()
#         and "_hardreal_real" not in p.name.lower()
#     ]

#     random.shuffle(source_paths)

#     idx = next_index(TRAIN_MIX_DIR)
#     saved = 0

#     for src in tqdm(source_paths, desc="Creating augreal images"):
#         if saved >= target_count:
#             break

#         img = cv2.imread(str(src))
#         if img is None:
#             continue

#         aug = degrade_real_image(img)
#         dst = TRAIN_MIX_DIR / f"{idx:04d}_augreal_real.jpg"

#         if cv2.imwrite(str(dst), aug):
#             saved += 1
#             idx += 1

#     print(f"Created augreal images: {saved}/{target_count}")


# create_augmented_reals(TARGET_REAL["augreal"])

# count_distribution(TRAIN_MIX_DIR)

In [ ]:
# Cell 10 — Create hard real images: 120 real

# def make_hard_real(img):
#     """
#     Stronger degradation recipes for real images:
#     dark, blurry, noisy, compressed, low-detail, or overexposed.
#     """
#     out = img.copy()

#     recipe = random.choice([
#         "dark_blur_noise",
#         "bright_blur_noise",
#         "grainy_compressed",
#         "low_detail_compressed",
#     ])

#     if recipe == "dark_blur_noise":
#         k = random.choice([3, 5, 7])
#         out = cv2.GaussianBlur(out, (k, k), 0)

#         alpha = random.uniform(0.45, 0.8)
#         out = np.clip(out.astype(np.float32) * alpha, 0, 255).astype(np.uint8)

#         noise = np.random.normal(0, random.uniform(8, 18), out.shape).astype(np.float32)
#         out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)

#     elif recipe == "bright_blur_noise":
#         k = random.choice([3, 5])
#         out = cv2.GaussianBlur(out, (k, k), 0)

#         alpha = random.uniform(1.1, 1.45)
#         out = np.clip(out.astype(np.float32) * alpha, 0, 255).astype(np.uint8)

#         noise = np.random.normal(0, random.uniform(6, 14), out.shape).astype(np.float32)
#         out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)

#     elif recipe == "grainy_compressed":
#         noise = np.random.normal(0, random.uniform(10, 22), out.shape).astype(np.float32)
#         out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)

#     elif recipe == "low_detail_compressed":
#         h, w = out.shape[:2]
#         down = cv2.resize(
#             out,
#             (max(32, w // 3), max(32, h // 3)),
#             interpolation=cv2.INTER_LINEAR
#         )
#         out = cv2.resize(down, (w, h), interpolation=cv2.INTER_LINEAR)
#         out = cv2.GaussianBlur(out, (3, 3), 0)

#     # Strong JPEG compression
#     q = random.randint(20, 55)
#     ok, enc = cv2.imencode(".jpg", out, [int(cv2.IMWRITE_JPEG_QUALITY), q])
#     if ok:
#         out = cv2.imdecode(enc, cv2.IMREAD_COLOR)

#     return out


# def create_hard_reals(target_count=120):
#     source_paths = [
#         p for p in TRAIN_MIX_DIR.iterdir()
#         if p.is_file()
#         and "_real" in p.name.lower()
#         and "_hardreal_real" not in p.name.lower()
#         and "_augreal_real" not in p.name.lower()
#     ]

#     random.shuffle(source_paths)

#     idx = next_index(TRAIN_MIX_DIR)
#     saved = 0

#     for src in tqdm(source_paths, desc="Creating hardreal images"):
#         if saved >= target_count:
#             break

#         img = cv2.imread(str(src))
#         if img is None:
#             continue

#         hard = make_hard_real(img)
#         dst = TRAIN_MIX_DIR / f"{idx:04d}_hardreal_real.jpg"

#         if cv2.imwrite(str(dst), hard):
#             saved += 1
#             idx += 1

#     print(f"Created hardreal images: {saved}/{target_count}")


# create_hard_reals(TARGET_REAL["hardreal"])

# count_distribution(TRAIN_MIX_DIR)

In [ ]:
# Cell 11 — Create degraded fake images: 200 fake

# def degrade_fake_image(img):
#     """
#     Degraded fake images designed to expose the model to difficult fake samples:
#     blur, noise, darkness, overexposure, compression, and low-detail artifacts.
#     """
#     out = img.copy()

#     # Blur
#     if random.random() < 0.6:
#         k = random.choice([3, 5, 7])
#         out = cv2.GaussianBlur(out, (k, k), 0)

#     # Noise
#     if random.random() < 0.5:
#         noise = np.random.normal(0, random.uniform(5, 25), out.shape).astype(np.float32)
#         out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)

#     # Darkening
#     if random.random() < 0.4:
#         out = np.clip(out.astype(np.float32) * random.uniform(0.3, 0.6), 0, 255).astype(np.uint8)

#     # Brightening / overexposure
#     if random.random() < 0.3:
#         out = np.clip(out.astype(np.float32) * random.uniform(1.4, 2.0), 0, 255).astype(np.uint8)

#     # JPEG compression
#     if random.random() < 0.8:
#         q = random.randint(20, 60)
#         ok, enc = cv2.imencode(".jpg", out, [int(cv2.IMWRITE_JPEG_QUALITY), q])
#         if ok:
#             out = cv2.imdecode(enc, cv2.IMREAD_COLOR)

#     return out


# def create_degraded_fakes(target_count=200):
#     source_paths = [
#         p for p in TRAIN_MIX_DIR.iterdir()
#         if p.is_file()
#         and "_fake" in p.name.lower()
#         and "_degradedfake_fake" not in p.name.lower()
#     ]

#     random.shuffle(source_paths)

#     idx = next_index(TRAIN_MIX_DIR)
#     saved = 0

#     for src in tqdm(source_paths, desc="Creating degradedfake images"):
#         if saved >= target_count:
#             break

#         img = cv2.imread(str(src))
#         if img is None:
#             continue

#         deg = degrade_fake_image(img)
#         dst = TRAIN_MIX_DIR / f"{idx:04d}_degradedfake_fake.jpg"

#         if cv2.imwrite(str(dst), deg):
#             saved += 1
#             idx += 1

#     print(f"Created degradedfake images: {saved}/{target_count}")


# create_degraded_fakes(TARGET_FAKE["degradedfake"])

# count_distribution(TRAIN_MIX_DIR)

In [ ]:
# Cell 12 — Final count check against expected distribution

# expected_real = TARGET_REAL
# expected_fake = TARGET_FAKE

# real_counts, fake_counts = count_distribution(TRAIN_MIX_DIR)

# print("\n" + "=" * 60)
# print("Checking final distribution against expected targets")
# print("=" * 60)

# ok = True

# print("\nREAL:")
# for source, expected in expected_real.items():
#     actual = real_counts.get(source, 0)
#     status = "OK" if actual == expected else "MISMATCH"
#     print(f"{source:20s} expected={expected:4d} actual={actual:4d}  {status}")
#     if actual != expected:
#         ok = False

# print("\nFAKE:")
# for source, expected in expected_fake.items():
#     actual = fake_counts.get(source, 0)
#     status = "OK" if actual == expected else "MISMATCH"
#     print(f"{source:20s} expected={expected:4d} actual={actual:4d}  {status}")
#     if actual != expected:
#         ok = False

# total_real = sum(real_counts.values())
# total_fake = sum(fake_counts.values())

# print("\nTOTAL:")
# print("Real:", total_real)
# print("Fake:", total_fake)
# print("Total:", total_real + total_fake)

# if ok and total_real == 1720 and total_fake == 1720:
#     print("\nFinal dataset matches expected 3440-image balanced training set.")
# else:
#     print("\nCheck mismatches above before training.")

In [ ]:
# Cell 13 — Save train.json, val.json, and test.json

# def label_from_filename(path):
#     name = Path(path).name.lower()

#     if "_real" in name:
#         return 0
#     if "_fake" in name:
#         return 1

#     return None


# def collect_train_items_from_mix(train_dir):
#     train_dir = Path(train_dir)
#     items = []

#     for p in sorted(train_dir.iterdir()):
#         if not p.is_file() or not is_image_file(p):
#             continue

#         label = label_from_filename(p)

#         if label is not None:
#             items.append({
#                 "path": str(p),
#                 "label": int(label)
#             })

#     random.shuffle(items)
#     return items


# def collect_items_with_reference(data_dir, ref_path):
#     data_dir = Path(data_dir)
#     ref_path = Path(ref_path)

#     with open(ref_path) as f:
#         labels = [int(line.strip()) for line in f if line.strip()]

#     images = sorted(glob.glob(str(data_dir / "*.*")))
#     images = [p for p in images if is_image_file(p)]

#     assert len(images) == len(labels), (
#         f"Mismatch: {len(images)} images vs {len(labels)} labels\n"
#         f"data_dir={data_dir}\n"
#         f"ref_path={ref_path}"
#     )

#     items = [
#         {"path": p, "label": int(l)}
#         for p, l in zip(images, labels)
#     ]

#     return items


# train_items = collect_train_items_from_mix(TRAIN_MIX_DIR)
# val_items = collect_items_with_reference(VAL_DIR, VAL_REF)
# test_items = collect_items_with_reference(TEST_DIR, TEST_REF)

# for name, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
#     n_real = sum(1 for x in items if x["label"] == 0)
#     n_fake = sum(1 for x in items if x["label"] == 1)

#     print(f"{name.upper()}: {len(items)} images  real={n_real}, fake={n_fake}")

#     out_path = STAGING / f"{name}.json"
#     with open(out_path, "w") as f:
#         json.dump(items, f, indent=2)

#     print(f"Saved: {out_path}\n")

In [ ]:
# Cell 14 — Optional visual sanity check

# import matplotlib.pyplot as plt
# from PIL import Image

# def show_random_samples(items, title, n=12):
#     sample = random.sample(items, min(n, len(items)))

#     cols = 6
#     rows = int(np.ceil(len(sample) / cols))

#     fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 2.4))
#     axes = np.array(axes).reshape(-1)

#     for ax in axes:
#         ax.axis("off")

#     for ax, item in zip(axes, sample):
#         img = Image.open(item["path"]).convert("RGB")
#         ax.imshow(img)
#         ax.set_title("real" if item["label"] == 0 else "fake", fontsize=9)
#         ax.axis("off")

#     plt.suptitle(title, fontsize=14, fontweight="bold")
#     plt.tight_layout()
#     plt.show()


# show_random_samples(train_items, "Random samples from expanded training set", n=12)